In [24]:
import yfinance as yf
import numpy as np
from scipy.stats import norm, chi2

alpha = 0.97
beta = 0.02
V = 1_000_000
M = 125_000
rng = np.random.default_rng(42)

# (a)


In [25]:
data = yf.download(
    "AAPL",
    start="2016-11-01",
    end="2018-11-01",   # end is exclusive
    auto_adjust=True,
    progress=False
)

prices = data["Close"].squeeze()

log_returns = np.log(
    prices / prices.shift(1)
).dropna()

n = len(log_returns)

X_bar = log_returns.mean()
S2 = log_returns.var(ddof=1)
S = np.sqrt(S2)

print("Number of prices:", len(prices))
print("Number of returns:", n)
print("Sample mean X_bar:", X_bar)
print("Sample variance S_n^2:", S2)
print("Sample std S_n:", S)

Number of prices: 504
Number of returns: 503
Sample mean X_bar: 0.0014055463103870146
Sample variance S_n^2: 0.0001671448309414722
Sample std S_n: 0.012928450446262776


In [26]:
# (a)(1)
losses = -V * (np.exp(log_returns.to_numpy()) - 1)
sorted_losses = np.sort(losses)
index = int(np.ceil(n * alpha)) - 1
var_empirical = sorted_losses[index]
print("Empirical VaR:", var_empirical)

Empirical VaR: 24984.708539570354


In [27]:
# (a)(2)
z_tail = norm.ppf(1 - alpha)
var_theoretical = V * (
    1 - np.exp(X_bar + S * z_tail)
)
print("Theoretical VaR:", var_theoretical)

Theoretical VaR: 22649.754752691264


In [28]:
# (a)(3)
q_low = chi2.ppf(beta / 2, df=n - 1)
q_high = chi2.ppf(1 - beta / 2, df=n - 1)

sigma_lower = S * np.sqrt((n - 1) / q_high)
sigma_upper = S * np.sqrt((n - 1) / q_low)

var_lower = V * (
    1 - np.exp(X_bar + sigma_lower * z_tail)
)

var_upper = V * (
    1 - np.exp(X_bar + sigma_upper * z_tail)
)

print("Lower sigma:", sigma_lower)
print("Upper sigma:", sigma_upper)

print(
    "98% VaR confidence interval:",
    var_lower,
    var_upper
)

Lower sigma: 0.01204187660663173
Upper sigma: 0.01394885451267235
98% VaR confidence interval: 21018.700473690922 24523.65673836654


# （b）

In [29]:
chi2_samples = rng.chisquare(
    df=n - 1,
    size=M
)
sigma_m = np.sqrt(
    (n - 1) * S2 / chi2_samples
)
mu_m = rng.normal(
    loc=X_bar,
    scale=sigma_m / np.sqrt(n),
    size=M
)
Y = V * (
    1 - np.exp(
        mu_m
        + sigma_m * norm.ppf(1 - alpha)
    )
)
Y_bar = np.mean(Y)
A = np.quantile(Y, beta / 2)
B = np.quantile(Y, 1 - beta / 2)

print(f"Average simulated VaR: ${Y_bar:,.2f}")
print(f"98% confidence interval: (${A:,.2f}, ${B:,.2f})")

Average simulated VaR: $22,685.74
98% confidence interval: ($20,583.57, $24,971.84)
